In [0]:
import mlflow 
mlflow.set_experiment('/Users/ananya.roy@databricks.com/freshmart_experiment')


## Agent evaluation via existing trace

In [0]:
import asyncio
import logging

import mlflow
from dotenv import load_dotenv
from mlflow.genai.agent_server import get_invoke_function
from mlflow.genai.scorers import (
    Completeness,
    ConversationalSafety,
    ConversationCompleteness,
    RelevanceToQuery,
    Safety,
    UserFrustration,
)
from mlflow.types.responses import ResponsesAgentRequest

# Load environment variables from .env if it exists
load_dotenv(dotenv_path=".env", override=True)
logging.getLogger("mlflow.utils.autologging_utils").setLevel(logging.ERROR)

In [0]:
# Retrieve existing traces from the experiment
# Suppress warnings for broken traces from failed simulation runs
logging.getLogger("mlflow.tracing.client").setLevel(logging.ERROR)

traces = mlflow.search_traces(
    filter_string="trace.status = 'OK'",
)
print(f"Found {len(traces)} traces")
traces

In [0]:
def evaluate():
    mlflow.genai.evaluate(
        data=traces,
        scorers=[
            Completeness(),
            ConversationCompleteness(),
            ConversationalSafety(),
            UserFrustration(),
            RelevanceToQuery(),
            Safety(),
            ToolCallEfficiency(),
        ],
    )
evaluate()


## Agent Evaluation via Calling App API

In [0]:

@mlflow.trace
def predict_fn(input: list[dict], **kwargs) -> dict:
    """Call the deployed Databricks App agent via HTTP."""
    # Step 1: Get your app's OAuth client ID
    w = WorkspaceClient()
    app_name = "agent-test-app"   
    app_client_id = w.apps.get(app_name).oauth2_app_client_id

    # Step 2: Get the notebook's internal token
    notebook_token = (
        dbutils.notebook.entry_point.getDbutils()
        .notebook().getContext().apiToken().get()
    )
    # Step 3: Exchange it for an audience-scoped OAuth token
    workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
    token_response = requests.post(
        url=f"https://{workspace_url}/oidc/v1/token",
        data={
            "grant_type": "urn:ietf:params:oauth:grant-type:token-exchange",
            "subject_token": notebook_token,
            "subject_token_type": "urn:databricks:params:oauth:token-type:personal-access-token",
            "requested_token_type": "urn:ietf:params:oauth:token-type:access_token",
            "scope": "all-apis",
            "audience": app_client_id,
        },
    )
    audience_token = token_response.json()["access_token"]

    response = requests.post(
        f"{APP_URL}/responses",  # Adjust path based on your agent's API
        headers={
            "Authorization": f"Bearer {audience_token}",
            "Content-Type": "application/json",
        },
        json={"input": input},
    )
    response.raise_for_status()
    return response.json()

eval_data = [
    {"inputs": {"input": [{"role": "user", "content": "How much revenue loss is tied with Return or refund and what does our refund policy says about it ? "}]}},
]

results = mlflow.genai.evaluate(
    data=eval_data,
    predict_fn=predict_fn,
    scorers=[RelevanceToQuery(), 
             Safety(), 
             ToolCallEfficiency()
             ],
)
results